# Лабораторная работа №1
## Предобработка данных. Знакомство с pandas
**Датасет:** Titanic (Kaggle)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1. Загрузка данных и вывод на экран

In [ ]:
df = pd.read_csv('titanic.csv')
print(f'Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов')
print(f'Столбцы: {df.columns.tolist()}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.dtypes

## 2. Количество пропущенных значений для каждого столбца

In [ ]:
print('=== Пропущенные значения ДО заполнения ===')
nan_counts = df.isnull().sum()
print(nan_counts)
print(f'\nВсего пропущенных значений: {nan_counts.sum()}')

## 3. Заполнение пропущенных значений
- **Age** (числовой) — заполняем медианой (устойчива к выбросам)
- **Cabin** (категориальный) — заполняем модой (самое частое значение)
- **Embarked** (категориальный) — заполняем модой

In [ ]:
# Медиана для числового столбца Age
age_median = df['Age'].median()
print(f'Медиана Age: {age_median}')
df['Age'].fillna(age_median, inplace=True)

# Мода для категориального столбца Cabin
cabin_mode = df['Cabin'].mode()[0]
print(f'Мода Cabin: {cabin_mode}')
df['Cabin'].fillna(cabin_mode, inplace=True)

# Мода для категориального столбца Embarked
embarked_mode = df['Embarked'].mode()[0]
print(f'Мода Embarked: {embarked_mode}')
df['Embarked'].fillna(embarked_mode, inplace=True)

In [ ]:
print('=== Пропущенные значения ПОСЛЕ заполнения ===')
print(df.isnull().sum())
print(f'\nВсего пропущенных значений: {df.isnull().sum().sum()}')

Все пропущенные значения успешно заполнены (0 пропусков).

## 4. Нормализация данных
Применяем MinMaxScaler к числовым столбцам (значения преобразуются в диапазон от 0 до 1).

In [ ]:
# Убираем столбцы, которые не несут полезной информации для модели
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis='columns')
print('Оставшиеся столбцы:', df.columns.tolist())
df.head()

In [ ]:
# Выделяем числовые столбцы (кроме целевого Survived)
numeric_cols = df.select_dtypes(include='number').columns.drop('Survived')
print('Числовые столбцы для нормализации:', numeric_cols.tolist())

scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print('\n=== Данные после нормализации ===')
df.head(10)

## 5. Преобразование категориальных данных (One-Hot Encoding)
Применяем OHE с `drop_first=True`, чтобы избежать мультиколлинеарности и переобучения модели.

In [ ]:
# Находим категориальные столбцы
object_cols = df.select_dtypes(include='object').columns.tolist()
print('Категориальные столбцы для OHE:', object_cols)

# Применяем One-Hot Encoding
df = pd.get_dummies(df, columns=object_cols, drop_first=True)

print(f'\nИтоговый размер датасета: {df.shape}')
print('Столбцы:', df.columns.tolist())
df.head(10)

## Разбиение на обучающую и тестовую выборки

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)
print(f'Обучающая выборка: {train_df.shape}')
print(f'Тестовая выборка: {test_df.shape}')

## Сохранение обработанных данных

In [ ]:
df.to_csv('processed_titanic.csv', index=False)
print('Обработанный датасет сохранён в processed_titanic.csv')